# ACE++ Option B Training Scaffold

This notebook trains the text-injection world-modeling policy with decoupled rewards. It is intentionally compact so it can run on a T4 with Qwen2.5-3B 4-bit, while the demo uses `demo_gradio.py`.

In [ ]:
!pip -q install -U "trl>=0.17.0" "transformers>=4.47.0" "accelerate>=0.34.0" "datasets>=2.20.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "matplotlib>=3.7" unsloth


In [ ]:
import importlib, json, random, sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

torch = importlib.import_module('torch')
Dataset = importlib.import_module('datasets').Dataset
set_seed = importlib.import_module('transformers').set_seed
trl = importlib.import_module('trl')
GRPOConfig = getattr(trl, 'GRPOConfig')
GRPOTrainer = getattr(trl, 'GRPOTrainer')
FastLanguageModel = importlib.import_module('unsloth').FastLanguageModel

from ace_world_env import WorldState
from ace_agents import AGENT_PROFILES
from ace_reward import compute_total_reward
from ace_env_openenv import ACEOpenMultiAgentEnv

set_seed(42)
random.seed(42)
assert torch.cuda.is_available(), 'Use a GPU runtime.'


In [ ]:
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 768

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
    device_map='auto',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.config.use_cache = False
print('Loaded', MODEL_NAME)


In [ ]:
def make_world(i, n, rng):
    ws = WorldState()
    progress = i / max(1, n)
    if progress < 0.25:
        ws.apply_deltas({'oil_price': rng.uniform(-0.2, 0.2), 'market_volatility': rng.uniform(-0.1, 0.1)})
    elif progress < 0.75:
        ws.apply_deltas({'oil_price': rng.uniform(-0.5, 0.8), 'trade_tension': rng.uniform(0, 0.4), 'market_volatility': rng.uniform(0, 0.3), 'cooperation_index': rng.uniform(-0.3, 0.3), 'resource_scarcity': rng.uniform(0, 0.4)})
    else:
        crisis = rng.choice(['oil', 'trade', 'financial', 'cooperation'])
        if crisis == 'oil':
            ws.apply_deltas({'oil_price': rng.uniform(0.5, 1.2), 'energy_cost': rng.uniform(0.3, 0.7), 'trade_tension': rng.uniform(0.2, 0.5)})
        elif crisis == 'trade':
            ws.apply_deltas({'trade_tension': rng.uniform(0.4, 0.7), 'cooperation_index': -rng.uniform(0.3, 0.5), 'market_volatility': rng.uniform(0.2, 0.5)})
        elif crisis == 'financial':
            ws.apply_deltas({'market_volatility': rng.uniform(0.4, 0.6), 'gdp_growth': -rng.uniform(0.04, 0.1), 'gold_price': rng.uniform(0.3, 0.8)})
        else:
            ws.apply_deltas({'cooperation_index': rng.uniform(0.3, 0.5), 'trade_tension': -rng.uniform(0.2, 0.4), 'market_volatility': -rng.uniform(0.1, 0.3)})
    return ws

def format_prompt_v2(observation, world_state, agent_id):
    profile = AGENT_PROFILES[agent_id % len(AGENT_PROFILES)]
    system = profile.system_prompt(world_state.to_prompt_str())
    user = '\n'.join([
        f'Agent ID: {agent_id}',
        'Recent interaction history:', json.dumps(observation.get('history', [])[-3:], indent=2),
        'Current trust scores:', json.dumps(observation.get('trust', {}), indent=2),
        'Current alliances:', json.dumps(observation.get('alliances', []), indent=2),
        'Output ONLY valid compact JSON with predicted_round, action, parameters, reasoning.'
    ])
    messages = [{'role':'system','content':system}, {'role':'user','content':user}]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return system + '\n\n' + user + '\nJSON:\n'

def build_prompt_dataset_v2(n_samples=360, seed=123):
    rng = random.Random(seed)
    rows = []
    for i in range(n_samples):
        ws = make_world(i, n_samples, rng)
        ground_truth = ws.sample_round_type(rng)
        difficulty = 'easy' if i / n_samples < 0.25 else ('medium' if i / n_samples < 0.75 else 'hard')
        env = ACEOpenMultiAgentEnv(num_agents=4, num_rounds=1, seed=seed+i, round_type_schedule=[ground_truth], difficulty=difficulty, id_shuffle=False, god_mode=True)
        obs = env.reset()
        state = env.state()
        agent_id = i % 4
        rows.append({'prompt': format_prompt_v2(obs, ws, agent_id), 'ground_truth': ground_truth, 'payoff_seed': int(state['current_payoff_seed']), 'difficulty': difficulty, 'agent_id': agent_id, 'num_agents': 4})
    return Dataset.from_list(rows)

train_dataset = build_prompt_dataset_v2(360, 123)
eval_dataset = build_prompt_dataset_v2(72, 999)
train_dataset[0]


In [ ]:
def extract_json(text):
    decoder = json.JSONDecoder()
    for idx, ch in enumerate(text):
        if ch != '{':
            continue
        try:
            obj, _ = decoder.raw_decode(text[idx:])
            return obj if isinstance(obj, dict) else {}
        except json.JSONDecodeError:
            pass
    return {}

def safe_parse_completion(text):
    obj = extract_json(text)
    predicted = obj.get('predicted_round') if obj.get('predicted_round') in ['cooperative','competitive','resource'] else 'resource'
    action = obj.get('action') if obj.get('action') else 'allocate_resources'
    params = obj.get('parameters') if isinstance(obj.get('parameters'), dict) else {'amount': 50}
    return {'predicted_round': predicted, 'action': action, 'parameters': params}, bool(obj)

reward_log_v2 = []

def reward_func_v2(prompts, completions, ground_truth, payoff_seed, difficulty, agent_id=None, num_agents=None, trainer_state=None, **kwargs):
    rewards, infs, acts, fmts = [], [], [], []
    agent_id = agent_id or [0] * len(completions)
    step = getattr(trainer_state, 'global_step', len(reward_log_v2))
    inference_w = max(0.5, 1.0 - (step / 300) * 0.5)
    action_w = min(0.8, 0.3 + (step / 300) * 0.5)
    for completion, gt, aid in zip(completions, ground_truth, agent_id):
        parsed, valid = safe_parse_completion(completion)
        profile = AGENT_PROFILES[int(aid) % len(AGENT_PROFILES)]
        r = compute_total_reward(completion, parsed['predicted_round'], parsed['action'], parsed['parameters'], gt, valid, profile, inference_weight=inference_w, action_weight=action_w, format_weight=0.25)
        rewards.append(r['total']); infs.append(r['inference']); acts.append(r['action']); fmts.append(r['format'])
    n = max(1, len(rewards))
    reward_log_v2.append({'step': step, 'avg_reward': sum(rewards)/n, 'inference_reward': sum(infs)/n, 'action_reward': sum(acts)/n, 'format_reward': sum(fmts)/n, 'accuracy': sum(infs)/n, 'inference_weight': inference_w, 'action_weight': action_w})
    return rewards


In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = GRPOConfig(output_dir='ace_grpo_v2_outputs', learning_rate=1e-5, per_device_train_batch_size=1, gradient_accumulation_steps=4, num_generations=4, max_steps=300, logging_steps=1, save_steps=150, save_strategy='steps', report_to='none', remove_unused_columns=False, bf16=bf16_supported, fp16=not bf16_supported, gradient_checkpointing=True, max_completion_length=128, temperature=0.9, top_p=0.95)
trainer = GRPOTrainer(model=model, args=training_args, reward_funcs=[reward_func_v2], train_dataset=train_dataset, eval_dataset=eval_dataset, processing_class=tokenizer)
trainer.train()


In [ ]:
import matplotlib.pyplot as plt
steps = [r['step'] for r in reward_log_v2]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(steps, [r['avg_reward'] for r in reward_log_v2]); axes[0,0].set_title('Total reward')
axes[0,1].plot(steps, [r['inference_reward'] for r in reward_log_v2], color='seagreen'); axes[0,1].axhline(0.333, color='gray', linestyle=':'); axes[0,1].set_title('Inference reward')
axes[1,0].plot(steps, [r['action_reward'] for r in reward_log_v2], color='coral'); axes[1,0].set_title('Action reward')
axes[1,1].plot(steps, [r['format_reward'] for r in reward_log_v2], color='purple'); axes[1,1].set_title('Format reward')
plt.tight_layout(); plt.savefig('training_curves_v2.png', dpi=150); plt.show()
